# 1. Entendimiento del negocio y de los datos

**Punto de la rubrica: a, c**

**Responsable sugerido:** Juan Pablo (persona A).

Este notebook responde: que hay realmente en los datos, que tan grande es, que tan limpio esta, y que necesitamos conseguir aparte.

## 1.1 Objetivo, usuario y unidad de analisis

**Pregunta de estudio:** ¿Que perfiles de demanda presentan las estaciones del componente troncal de TransMilenio segun sus patrones temporales de validaciones, y que estaciones (y en que horas de la semana) requieren prioridad en la gestion operativa -incluyendo personal de control de acceso/orden- durante las horas pico?

**Usuario:** gerencia de operaciones de TransMilenio / Secretaria de Movilidad - en particular quien decide contratacion y turnos de personal en estaciones.

**Unidad de analisis:** la estacion troncal, en una tabla intermedia estacion-fecha-hora (ver seccion 1.3 y notebook 02).

**Hipotesis:** (1) las estaciones se agrupan en pocos perfiles de demanda (residencial, destino laboral, transferencia, baja demanda); (2) la necesidad de personal de control se concentra en pocas estaciones y franjas horarias, no de forma pareja; (3) el patron de un festivo se parece mas al de un fin de semana que al de un dia habil.

Detalle completo y cronograma en `CLAUDE.md`.

## 1.2 Inventario del conjunto de datos

Ventana usada: 10-16 agosto, 14-20 septiembre (semanas tipicas, base del modelo) y 3-9 agosto (incluye el festivo del 7 de agosto, se usa solo como comparacion puntual, ver `CLAUDE.md`). Formato real: un ZIP por dia, cada uno con un unico CSV adentro.

In [1]:
import pandas as pd
import zipfile
from pathlib import Path

RAW = Path('../data/raw/validaciones_troncal')
archivos = sorted(RAW.glob('*.zip'))

resumen = []
for f in archivos:
    with zipfile.ZipFile(f) as z:
        info = z.infolist()[0]
        with z.open(info) as fh:
            n_filas = sum(1 for _ in fh) - 1  # -1 por el encabezado
    resumen.append({
        'archivo': f.name,
        'peso_zip_MB': round(f.stat().st_size / 1e6, 1),
        'peso_csv_MB': round(info.file_size / 1e6, 1),
        'filas': n_filas,
    })

inventario = pd.DataFrame(resumen)
inventario

                            archivo  peso_zip_MB  peso_csv_MB    filas
0   validacionTroncal20260803.zip        123.4        594.6  1817362
1   validacionTroncal20260804.zip        127.6        615.1  1879864
2   validacionTroncal20260805.zip        128.4        619.4  1893704
3   validacionTroncal20260806.zip        124.5        601.3  1837065
4   validacionTroncal20260807.zip         33.8        173.4   525817  <- festivo
5   validacionTroncal20260808.zip         69.9        349.3  1062209
6   validacionTroncal20260809.zip         32.7        170.4   516440
7   validacionTroncal20260810.zip        118.8        573.2  1751965
8   validacionTroncal20260811.zip        124.2        599.6  1832551
9   validacionTroncal20260812.zip        125.5        606.0  1852083
10  validacionTroncal20260813.zip        124.5        600.8  1837133
11  validacionTroncal20260814.zip        123.8        598.0  1827789
12  validacionTroncal20260815.zip         75.3        374.2  1139154
13  validacionTronca

In [1]:
print(f"Total: {inventario['filas'].sum():,} filas en {len(inventario)} archivos")
print(f"Peso total comprimido: {inventario['peso_zip_MB'].sum()/1024:.2f} GB")
print(f"Peso total CSV (sin comprimir): {inventario['peso_csv_MB'].sum()/1024:.2f} GB")

Total: 31,530,646 filas en 21 archivos
Peso total comprimido: 2.00 GB
Peso total CSV (sin comprimir): 9.65 GB

**Evidencia de escala:** 31.5 millones de registros, ~9.65 GB sin comprimir (supera varias veces el minimo de 1.5 GB pedido). Este es el numero que justifica usar Dask en el notebook 03 en vez de cargar todo en memoria con pandas.

## 1.3 Columnas reales y diccionario de datos

**Importante -leer con cuidado-:** al inspeccionar los datos reales aparecieron varias sorpresas frente a lo que dice la pagina de datos abiertos. Quedan documentadas aqui para que nadie del equipo pierda tiempo asumiendo lo contrario.

In [1]:
# Leer un archivo tipico completo, con la codificacion correcta (ver hallazgo de encoding abajo)
df = pd.read_csv(RAW / 'validacionTroncal20260915.zip', compression='zip', encoding='utf-8')
df.dtypes

Acceso_Estacion                   object
Day_Group_Type                    object
Dispositivo                        int64
Emisor                             object
Estacion_Parada                    object
Fase                               object
Fecha_Clearing                     object
Fecha_Transaccion                  object
Hora_Pico_SN                       object
ID_Vehiculo                      float64
Linea                              object
Nombre_Perfil                      object
Numero_Tarjeta                     object
Operador                           object
Ruta                              float64
Saldo_Despues_Transaccion        float64
Saldo_Previo_a_Transaccion       float64
Sistema                            object
Tipo_Tarifa                        object
Tipo_Tarjeta                       object
Tipo_Vehiculo                     float64
Valor                             float64
dtype: object

### Hallazgos de la inspeccion real (sobre `validacionTroncal20260915.zip`, 1,897,215 filas)

| Hallazgo | Detalle | Que hacer |
|---|---|---|
| **`Day_Group_Type` no sirve como viene** | Siempre vale `"Dia 1"` para TODAS las filas del archivo (no distingue habil/sabado/domingo como asumiamos en `CLAUDE.md`) | Derivar el tipo de dia nosotros mismos con `Fecha_Transaccion.dt.dayofweek` + una lista propia de festivos (ya sabemos que el 7 de agosto 2026 es festivo) |
| **`Hora_Pico_SN` no sirve como viene** | Siempre vale `"Peak Time"` para todas las filas | Derivar hora pico nosotros mismos: AM 5-9h, PM 16-20h (regla ya definida en `CLAUDE.md`), usando la hora de `Fecha_Transaccion` |
| **`Sistema`** | Columna no documentada en el portal, pero siempre vale `"TRONCAL"` (confirma que es el subconjunto correcto) | Sin uso analitico, se puede descartar |
| **`ID_Vehiculo`, `Ruta`, `Tipo_Vehiculo` 100% nulos** | Confirmado: el portal ya avisaba que solo aplican a Zonal/Dual | Descartar estas 3 columnas para Troncal |
| **Codificacion: el archivo es UTF-8** | Probado con los bytes crudos (`Ã©` = 'e' con tilde en UTF-8, confirmado con `ord()`); usar `encoding='latin1'` (que se penso erroneamente al inicio) corrompe los acentos en 2 caracteres (ej. 'America' -> 'AmA ricas'). Leer siempre con `encoding='utf-8'` (que ademas es el valor por defecto de pandas, no hace falta ni especificarlo) |
| **"Sangrado" de fechas entre archivos consecutivos** | El archivo del 15-sep trae 262 filas del 14-sep y 8 filas del 16-sep (0.014%); son transacciones reales cerca de medianoche, no basura | Al unir varios dias, agrupar SIEMPRE por la fecha real de `Fecha_Transaccion`, no por el nombre del archivo de origen |
| **Sin duplicados entre dias** | Se comparo el archivo del 14-sep contra el del 15-sep (tarjeta+fecha exacta+estacion+dispositivo+valor): 0 filas repetidas | Se puede concatenar los 21 archivos sin riesgo de contar dos veces la misma validacion |
| **`Tipo_Tarifa` viene como texto con coma decimal** | Valores `'1,0'`, `'2,0'`, `'3,0'` (no son numeros, son codigos de tarifa con formato raro) | Tratar como categoria, no como numero; mapear a una etiqueta legible en la limpieza |
| **Numero de estaciones activas varia por dia** | 150 estaciones el 15-sep, 148 el 7-ago (festivo) | Normal (algunas estaciones no tuvieron ninguna validacion ese dia), no es un error |


### Diccionario de columnas (Validacion Troncal)

| Columna | Tipo real | Descripcion | Uso en el proyecto |
|---|---|---|---|
| `Acceso_Estacion` | texto | Acceso especifico de la estacion | Contexto, no clave |
| `Day_Group_Type` | texto (**inutil**, ver arriba) | - | Descartar, derivar tipo de dia propio |
| `Dispositivo` | entero | Identificador del torniquete/dispositivo | Contexto |
| `Emisor` | texto | Emisor de la tarjeta | Contexto |
| `Estacion_Parada` | texto | Estacion, formato `"(codigo) Nombre"` | **Clave para unir con el dataset de estaciones** |
| `Fase` | texto | Fase del sistema a la que pertenece | Contexto |
| `Fecha_Clearing` | texto/fecha | Fecha en que el registro llego al sistema | No usar para agregaciones por dia (usar `Fecha_Transaccion`) |
| `Fecha_Transaccion` | texto/timestamp | Momento real de la validacion | **Clave temporal: fecha, hora, dia de la semana** |
| `Hora_Pico_SN` | texto (**inutil**, ver arriba) | - | Descartar, derivar hora pico propia |
| `ID_Vehiculo` | 100% nulo | Solo aplica a Zonal/Dual | Descartar |
| `Linea` | texto | En troncal, la zona de la troncal | Feature posible |
| `Nombre_Perfil` | texto | Tipo de tarjeta (Adulto, Adulto Mayor, Discapacidad, etc.) | Feature posible (perfil socioeconomico de la demanda) |
| `Numero_Tarjeta` | texto | Numero de tarjeta (no anonimizado a nivel de hash) | Identificador de usuario; NO se sube a AWS (dato sensible) |
| `Operador` | texto | Operador del servicio | Contexto |
| `Ruta` | 100% nulo | Solo aplica a Zonal | Descartar |
| `Saldo_Previo_a_Transaccion`, `Saldo_Despues_Transaccion` | decimal | Saldo de la tarjeta antes/despues | No relevante para el analisis de demanda |
| `Sistema` | texto (constante `TRONCAL`) | Confirma el subconjunto | Descartar |
| `Tipo_Tarifa` | texto/categoria | Codigo de tarifa aplicada | Feature posible |
| `Tipo_Tarjeta` | texto | Tipo de tarjeta | Feature posible |
| `Tipo_Vehiculo` | 100% nulo | Solo aplica a Zonal | Descartar |
| `Valor` | decimal | Valor cobrado en la transaccion | Contexto, no la variable de demanda (la demanda es el CONTEO de filas, no `Valor`) |

**Nota de privacidad:** `Numero_Tarjeta` es informacion potencialmente sensible. No se usa para identificar personas, no se sube a AWS, y si se usa para algo (ej. contar tarjetas unicas por estacion) se descarta la columna inmediatamente despues.

## 1.4 Estaciones (lat/lon) y como cruzarlas

Tenemos 2 archivos de estaciones (`data/raw/estaciones/`), con columnas distintas (pendiente elegir cual usar como canonico, o combinar ambos). El campo clave para cruzar con las validaciones es el **codigo de estacion de 5 digitos**: en validaciones viene pegado al nombre como `"(05002) Tintal"`; en el dataset de estaciones viene aparte en la columna `numero_estacion`. La union se hace por codigo, no por nombre (evita problemas de tildes/mayusculas/abreviaturas).

In [1]:
# Ejemplo de como se ve el codigo pegado al nombre en las validaciones:
df['Estacion_Parada'].dropna().unique()[:5]

array(['(05002) Tintal', '(07000) Portal Sur JFK Coop. Financiera',
       '(15001) Tibanica - Primavera', '(10002) Av. Primero de Mayo',
       '(05100) Banderas P. Central'], dtype=object)

## 1.5 Informacion de contexto adicional (punto c)

- **Conseguido:** coordenadas de estaciones (lat/lon), calendario de festivos colombianos (verificado para agosto 2026: 7 de agosto es festivo).
- **Falta / no disponible:** capacidad y frecuencia real de buses por troncal (no esta en datos abiertos), eventos puntuales de esas semanas (marchas, cierres, conciertos), clima (IDEAM). Sin esto, el proyecto mide **presion de demanda (validaciones)**, no saturacion real del servicio (oferta vs demanda). Se documenta como limitacion en las conclusiones (Fase 5).
- **Tampoco tenemos:** informacion sobre donde hay hoy personal de control de acceso/orden en estaciones (para comparar contra lo que recomiende el modelo). Si alguien del equipo consigue esto de forma informal, seria una validacion valiosa del indice de prioridad (notebook 04).